# Advanced: Why Modern Hopfield Networks Converge

This notebook answers a focused question about the modern Hopfield model from L6c: why do continuous-state updates settle, and what does convergence mean here?

This is a proof-only notebook. For simulations and numerical checks, see the example notebook.

> __Learning Objectives:__
>
> By the end of this notebook, you should be able to:
>
> * __Derive the descent step:__ Starting from the continuous-state LSE energy and the softmax update map, show $\nabla g(\mathbf{s})=\mathbf{T}(\mathbf{s})$, build the quadratic surrogate $Q(\mathbf{u}\mid\mathbf{v})$, and prove the one-step decrease formula $E(\mathbf{T}(\mathbf{v}))\le E(\mathbf{v})-\tfrac12\lVert\mathbf{T}(\mathbf{v})-\mathbf{v}\rVert_2^2$.
> * __Use descent to prove convergence:__ Use the log-sum-exp bound and norm inequalities to get $E(\mathbf{s})\ge\tfrac12(\lVert\mathbf{s}\rVert_2-M)^2\ge0$, then sum the descent inequalities to show finite total squared movement and therefore $\lVert\mathbf{s}^{t+1}-\mathbf{s}^{t}\rVert_2\to0$.
> * __Characterize the endpoint:__ Show iterates are weighted averages of stored memories, so they stay in a bounded region and have limit points. Then use continuity of $\mathbf{T}$ and $\nabla E(\mathbf{s})=\mathbf{s}-\mathbf{T}(\mathbf{s})$ to prove every limit point is fixed/stationary.

Let's get started!

___


## Convergence

Before we dive into equations, keep the big picture in mind: each modern Hopfield update minimizes a local surrogate of the energy. That gives one-step descent. Descent plus a global lower bound gives asymptotic convergence.

> __Intuition:__
>
> The update map compares the current state with all stored memories, then replaces the state with a softmax-weighted average of those memories. The proof explains why this replacement lowers the energy.
>
> The endpoint is asymptotic, not finite-step: energy converges, update sizes go to zero, and limit points satisfy the fixed-point equation.

Before the theorem, we define four terms used in the statement.

> __Terminology (plain language, used below):__
> - **Limit point:** a value the iterates get arbitrarily close to infinitely many times.
> - **Fixed point:** a state unchanged by one update, i.e., $\mathbf{T}(\mathbf{s})=\mathbf{s}$.
> - **Stationary point:** a state where the gradient is zero, i.e., $\nabla E(\mathbf{s})=\mathbf{0}$.
> - **Compact set:** a closed and bounded set; sequences in it have at least one limit point.

With this terminology in place, we now state the theorem and then prove it step by step.

> __Convergence Theorem (Modern Hopfield, continuous-state LSE energy; self-contained notation):__
>
> Let $N,K\in\mathbb{Z}_{\geq 1}$, where $N$ is the state dimension (number of nodes/features) and $K$ is the number of stored memories. Let $\beta>0$ denote the inverse temperature parameter. Let
> $\mathbf{X}=[\mathbf{m}_1,\ldots,\mathbf{m}_K]\in\mathbb{R}^{N\times K}$ be the memory matrix, where column $\mathbf{m}_i\in\mathbb{R}^{N}$ is stored memory $i$. Define
> $M:=\max_{i=1,\ldots,K}\lVert\mathbf{m}_i\rVert_2$, i.e., $M$ is the largest memory norm.
>
> For any similarity vector $\mathbf{z}\in\mathbb{R}^{K}$ and any state $\mathbf{s}\in\mathbb{R}^{N}$, define
> $$
> \begin{aligned}
> \operatorname{lse}_{\beta}(\mathbf{z})
> &:= \frac{1}{\beta}\log\left(\sum_{i=1}^{K}e^{\beta z_i}\right),\\
> E(\mathbf{s})
> &:= -\operatorname{lse}_{\beta}(\mathbf{X}^{\top}\mathbf{s}) + \frac{1}{2}\lVert\mathbf{s}\rVert_2^2 + \frac{1}{\beta}\log K + \frac{1}{2}M^2,\\
> \mathbf{T}(\mathbf{s})
> &:= \mathbf{X}\,\operatorname{softmax}(\beta\mathbf{X}^{\top}\mathbf{s}).
> \end{aligned}
> $$
> Here $\operatorname{softmax}(\mathbf{u})_i := \frac{e^{u_i}}{\sum_{j=1}^{K}e^{u_j}}$ for $\mathbf{u}\in\mathbb{R}^{K}$ and $i=1,\ldots,K$. Also, $\operatorname{lse}_{\beta}$ is the smooth maximum over similarities, $E$ is the Lyapunov energy, and $\mathbf{T}$ is the retrieval update map. Let the iteration be $\mathbf{s}^{t+1}=\mathbf{T}(\mathbf{s}^{t})$ for $t=0,1,2,\ldots$. Then, for every $t$,
> $$
> E(\mathbf{s}^{t+1}) \le E(\mathbf{s}^{t}) - \frac{1}{2}\lVert\mathbf{s}^{t+1}-\mathbf{s}^{t}\rVert_2^2.
> $$
> Interpretation: each update decreases energy by at least half the squared update size.
> Also, for every $\mathbf{s}\in\mathbb{R}^{N}$,
> $$
> E(\mathbf{s}) \ge \frac{1}{2}\left(\lVert\mathbf{s}\rVert_2 - M\right)^2 \ge 0.
> $$
> Interpretation: this is a global energy floor, so energy cannot decrease without bound.
> Therefore $E(\mathbf{s}^{t})$ is monotone non-increasing and convergent, and
> $$
> \lVert\mathbf{s}^{t+1}-\mathbf{s}^{t}\rVert_2 \to 0.
> $$
> Interpretation: the change from one iterate to the next goes to zero.
> Every limit point $\mathbf{s}^{\ast}$ satisfies
> $$
> \mathbf{T}(\mathbf{s}^{\ast})=\mathbf{s}^{\ast},
> $$
> equivalently
> $$
> \nabla E(\mathbf{s}^{\ast})=\mathbf{0}.
> $$
> This theorem is asymptotic; it does not assert universal finite-step convergence.

Now we put the gradient-descent viewpoint at the center. Define:
$$
g(\mathbf{s}) := \operatorname{lse}_{\beta}(\mathbf{X}^{\top}\mathbf{s}).
$$
Here $g$ is the smooth similarity term in the energy. Then:
$$
\begin{align*}
\nabla g(\mathbf{s})
&= \mathbf{X}\,\operatorname{softmax}(\beta\mathbf{X}^{\top}\mathbf{s}) && \text{differentiate lse and apply chain rule}\\
&= \mathbf{T}(\mathbf{s}) && \text{use update-map definition},\\
\nabla E(\mathbf{s})
&= \mathbf{s}-\nabla g(\mathbf{s}) && \text{differentiate the energy}\\
&= \mathbf{s}-\mathbf{T}(\mathbf{s}) && \text{substitute the previous line}.
\end{align*}
$$
This is the key point: the self-attention map $\mathbf{T}(\mathbf{s})$ appears directly as a gradient term. So attention is not added on top of gradient descent; it is what gradient descent on this energy naturally computes.

A useful update family in practice is given by:
$$
\mathbf{s}^{t+1}=(1-\eta)\mathbf{s}^{t}+\eta\mathbf{T}(\mathbf{s}^{t}),\quad 0<\eta\leq 1.
$$
Using $\nabla E(\mathbf{s})=\mathbf{s}-\mathbf{T}(\mathbf{s})$, this is exactly:
$$
\mathbf{s}^{t+1}=\mathbf{s}^{t}-\eta\nabla E(\mathbf{s}^{t}).
$$
So the theorem's retrieval map is simply the step-size-$1$ case ($\eta=1$). Fixed points of $\mathbf{T}$ are therefore exactly stationary points of $E$.

With that interpretation established, the next step is to show the energy is non-increasing at each update. We do this by applying convexity of $g$ for arbitrary $\mathbf{u},\mathbf{v}\in\mathbb{R}^{N}$ and rearranging:
$$
\begin{align*}
g(\mathbf{u})
&\ge g(\mathbf{v}) + \nabla g(\mathbf{v})^{\top}(\mathbf{u}-\mathbf{v}) && \text{use convexity},\\
-g(\mathbf{u})
&\le -g(\mathbf{v}) - \nabla g(\mathbf{v})^{\top}(\mathbf{u}-\mathbf{v}) && \text{multiply both sides by }-1,\\
E(\mathbf{u})
&\le Q(\mathbf{u}\mid\mathbf{v})
:= \frac12\lVert\mathbf{u}\rVert_2^2 - \nabla g(\mathbf{v})^{\top}\mathbf{u} + c(\mathbf{v}) && \text{add the same quadratic and constants on both sides}.
\end{align*}
$$
Here $Q(\cdot\mid\mathbf{v})$ is the local quadratic surrogate evaluated at reference point $\mathbf{v}$, and $c(\mathbf{v})$ is independent of $\mathbf{u}$. The minimizer of $Q(\cdot\mid\mathbf{v})$ and its squared-distance decrease are:
$$
\begin{align*}
\mathbf{u}^{\ast}
&= \nabla g(\mathbf{v}) = \mathbf{T}(\mathbf{v}) && \text{set derivative of }Q\text{ to zero},\\
Q(\mathbf{u}^{\ast}\mid\mathbf{v})
&= Q(\mathbf{v}\mid\mathbf{v}) - \frac12\lVert\mathbf{u}^{\ast}-\mathbf{v}\rVert_2^2 && \text{complete the square}.
\end{align*}
$$
Combining $E(\mathbf{u}^{\ast})\le Q(\mathbf{u}^{\ast}\mid\mathbf{v})$ with $Q(\mathbf{v}\mid\mathbf{v})=E(\mathbf{v})$ gives:
$$
\begin{align*}
E(\mathbf{T}(\mathbf{v}))
&\le E(\mathbf{v}) - \frac12\lVert\mathbf{T}(\mathbf{v})-\mathbf{v}\rVert_2^2 && \text{one-step decrease formula}.
\end{align*}
$$
Interpretation: larger updates force larger energy drops, so persistent large jumps cannot continue. Setting $\mathbf{v}=\mathbf{s}^{t}$ gives the theorem's one-step descent statement. Next, derive the lower bound in one clean chain. Start with:
$$
\operatorname{lse}_{\beta}(\mathbf{z}) \le \max_i z_i + \frac{1}{\beta}\log K,
$$
and substitute $\mathbf{z}=\mathbf{X}^{\top}\mathbf{s}$:
$$
\begin{align*}
E(\mathbf{s})
&= -\operatorname{lse}_{\beta}(\mathbf{X}^{\top}\mathbf{s}) + \frac12\lVert\mathbf{s}\rVert_2^2 + \frac{1}{\beta}\log K + \frac12 M^2 && \text{definition of }E,\\
&\ge -\max_i \mathbf{m}_i^{\top}\mathbf{s} + \frac12\lVert\mathbf{s}\rVert_2^2 + \frac12 M^2 && \text{replace lse with its upper bound},\\
&\ge -M\lVert\mathbf{s}\rVert_2 + \frac12\lVert\mathbf{s}\rVert_2^2 + \frac12 M^2 && \text{bound dot product by product of norms},\\
&= \frac12\left(\lVert\mathbf{s}\rVert_2 - M\right)^2 \ge 0 && \text{complete the square}.
\end{align*}
$$
Interpretation: this lower bound gives an energy floor and rules out runaway descent. So energy is lower-bounded and, by descent, $E(\mathbf{s}^{t})$ converges. The same descent inequality also gives vanishing increments after summing across steps (the middle terms cancel):
$$
\begin{align*}
\frac12\sum_{t=0}^{T-1}\lVert\mathbf{s}^{t+1}-\mathbf{s}^{t}\rVert_2^2
&\le \sum_{t=0}^{T-1}\big(E(\mathbf{s}^{t})-E(\mathbf{s}^{t+1})\big) && \text{sum one-step decrease inequalities},\\
&= E(\mathbf{s}^{0})-E(\mathbf{s}^{T}) && \text{middle terms cancel},\\
&\le E(\mathbf{s}^{0}) && \text{use }E(\mathbf{s}^{T})\ge 0.
\end{align*}
$$
Letting $T\to\infty$, the series $\sum_t\lVert\mathbf{s}^{t+1}-\mathbf{s}^{t}\rVert_2^2$ is finite, so $\lVert\mathbf{s}^{t+1}-\mathbf{s}^{t}\rVert_2\to0$. Interpretation: finite total squared movement means update sizes must go to zero.

Finally, let's look at what happens to the sequence as it runs. For $t\ge1$, $\mathbf{s}^{t}=\mathbf{X}\mathbf{p}^{t}$ with $\mathbf{p}^{t}:=\operatorname{softmax}(\beta\mathbf{X}^{\top}\mathbf{s}^{t-1})$, so $\mathbf{p}^{t}$ is a probability vector (all entries nonnegative and sum to one). That means each $\mathbf{s}^{t}$ is a weighted average of the stored memories, and all iterates stay inside the convex hull of the memories. Because this region is closed and bounded, the sequence can't escape and must have at least one value it gets arbitrarily close to over and over (a limit point). Pick a subsequence $t_n$ so that $\mathbf{s}^{t_n}\to\mathbf{s}^{\ast}$. Since the step sizes vanish, $\mathbf{s}^{t_n+1}-\mathbf{s}^{t_n}\to0$, so $\mathbf{s}^{t_n+1}\to\mathbf{s}^{\ast}$ as well. By continuity of $\mathbf{T}$:
$$
\begin{align*}
\mathbf{T}(\mathbf{s}^{\ast})
&= \lim_{n\to\infty}\mathbf{T}(\mathbf{s}^{t_n}) && \text{use continuity of update map},\\
&= \lim_{n\to\infty}\mathbf{s}^{t_n+1} && \text{replace using iteration rule},\\
&= \mathbf{s}^{\ast} && \text{shifted sequence has same limit}.
\end{align*}
$$
So every limit point is fixed, and therefore stationary by $\nabla E(\mathbf{s})=\mathbf{s}-\mathbf{T}(\mathbf{s})$.

> __Proof conclusion:__
>
> Modern Hopfield retrieval is an energy-descent process for the continuous LSE model. The descent gap controls update size, update sizes vanish asymptotically, and every limit point is a fixed/stationary state.

___

## Summary

This notebook gives a proof-only convergence result for the modern Hopfield model used in L6c.

> __Key Takeaways:__
>
> * __Update rule and energy are directly linked:__ In this model, the softmax update map is the gradient of the smooth similarity term, so $\nabla E(\mathbf{s})=\mathbf{s}-\mathbf{T}(\mathbf{s})$. This identity enables the surrogate argument and gives a quantitative one-step descent inequality.
> * __Asymptotic settling comes from descent plus a lower bound:__ The proof gives both per-step decrease and an explicit global lower bound on energy. Summing the descent inequalities yields finite cumulative squared movement, which forces update increments to vanish.
> * __Convergence here means fixed/stationary limit points:__ Because iterates are weighted averages of memories, trajectories remain in a compact region and limit points exist. Every limit point satisfies $\mathbf{T}(\mathbf{s}^{\ast})=\mathbf{s}^{\ast}$ (equivalently $\nabla E(\mathbf{s}^{\ast})=\mathbf{0}$).

___


## References
* Ramsauer H, Schafl B, Lehner J, et al. (2021), *Hopfield Networks is All You Need*, ICLR 2021. arXiv: https://arxiv.org/abs/2008.02217. Local copy in this repo: `lectures/week-6/L6c/docs/Ramsauer-HNAYN-2021.pdf`

* Krotov D, Hopfield JJ (2021), *Large Associative Memory Problem in Neurobiology and Machine Learning*, ICLR 2021. arXiv: https://arxiv.org/abs/2008.06996. Local copy in this repo: `lectures/week-6/L6c/docs/Krotov-Hopfield-2021.pdf`

* Krotov D, Hopfield JJ (2016), *Dense Associative Memory for Pattern Recognition*, NeurIPS 2016. arXiv: https://arxiv.org/abs/1606.01164. Local copy in this repo: `lectures/week-6/L6c/docs/Krotov-Hopfield-2016.pdf`

* Demircigil M, Heusel J, Lowe M, Upgang S, Vermet F (2017), *On a Model of Associative Memory with Huge Storage Capacity*, Journal of Statistical Physics. arXiv: https://arxiv.org/abs/1702.01929. Local copy in this repo: `lectures/week-6/L6c/docs/Vermet-2017.pdf`

____